# AgroSynapse Yield Prediction

Baseline model for predicting yield from cleaned survey data.

In [ ]:
import pandas as pd
from pandas.api.types import is_bool_dtype, is_numeric_dtype
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = 'data/processed/clean_data.csv'
TARGET = 'yield_tons'
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=[TARGET]) if TARGET in df.columns else df
df.head()

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f'Target column {TARGET} not found in dataset.')

X = df.drop(columns=[TARGET])
y = df[TARGET]

categorical_cols = []
numeric_cols = []
for column in X.columns:
    series = X[column]
    if is_numeric_dtype(series) and not is_bool_dtype(series):
        numeric_cols.append(column)
    else:
        categorical_cols.append(column)

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', categorical_pipeline, categorical_cols),
        ('numeric', numeric_pipeline, numeric_cols)
    ],
    remainder='drop'
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = mean_squared_error(y_test, preds, squared=False)
{'mae': mae, 'rmse': rmse}